In [ ]:
# 本文所采用的方案： https://blog.csdn.net/qq_41813454/article/details/129906100
#  bili李牧视频方案: https://www.bilibili.com/video/BV16g411L7FG/?p=2&spm_id_from=pageDriver&vd_source=f6028d67767f2407fd726d6dfcc837cc
import torch.nn as nn

In [4]:
class Encoder(nn.Module):
    def __init__(self,
                 input_dim,
                 emb_dim,
                 hid_dim,
                 n_layers,
                 dropout
                 ):
        super().__init__()
        self.hid_dim = hid_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(input_dim,emb_dim)
        self.lstm = nn.LSTM(emb_dim, # 词嵌入后，每个词空间维度
                            hid_dim,
                            n_layers,
                            dropout=dropout,
                            batch_first=True)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, src):
       # src = (batch_size, src_len)
       # embedded = (batch_size, src_len, emb_dim)
       embedded = self.dropout(self.embedding(src))
       outputs, (hidden,cell) = self.lstm(embedded) 
       # 根据outputs的shape,可以看出，多层RNN的输出串联到下一层的输入，多向RNN的输出结果是cat的
       # outputs = (batch_size, src_len, hid_dim*n_directions) 
       
       # hidden = (n_layers*n_directions,batch_size,hid_dim)  
       # cell = (n_layers*n_directions,batch_size,hid_dim) 
       return hidden, cell

In [6]:
class Decoder(nn.Module):
    def __init__(self,
                 output_dim,
                 emb_dim,
                 hid_dim,
                 n_layers,
                 dropout):
        self.output_dim = output_dim
        self.hid_dim = hid_dim
        self.n_layers = n_layers
        self.embedding = nn.Embdding(output_dim,emb_dim)
        self.lstm = nn.LSTM(emb_dim,hid_dim,n_layers,dropout,
                            batch_first=True
                            )
        self.fc_out = nn.Linear(hid_dim,output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, input, hidden, cell):
        # 各输入的形状
        # input = (batch_size)
        # hidden = (n_layers*n_directions,batch_s,hid_dim)
        # cell = (n_layers*n_directions,batch_s,hid_dim)
        
        # LSTM是单向的 ==> n_directions = 1
        # hidden=(n_layers,batch_size,hid_dim)
        # cell=(n_layers,batch_size,hid_dim)
        
        input = input.unsequeeze(1) # (batch)-->(batch,1) 让输入的长度为1
        embedded = self.dropout(self.embeding(input)) #  # (batch_size,1, emb_dim)
        
        output,(hidden,cell) = self.lstm(embedded,(hidden,cell))
        
        # LSTM理论上的输出形状
        # output=(batch_size,seq_len,hid_dim*n_directions)
        # hidden=(num_layers*n_directions,batch_size,hid_dim)
        
        # 解码器中的序列长度 seq_len=1
        # 且其是单向 n_directions=1
        # output=(batch_size,1,hid_dim)
        # hidden=(n_layers,batch_size,hid_dim)
        #   cell=(n_layers,batch_size,hid_dim)
        prediction=self.fc_out(output.squeeze(1))
        # prediction=(batch_size,output_dim)
        return prediction, hidden, cell       

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self):
        self.encoder = Encoder()
        self.decoder = Decoder()
        self.device = device
        
    def forward(self,src,trg,teacher_forcing_ratio=0.5):
        # src = (batch_size,src_len)
        # trg = (batch_szie,trg_len)
        # teacher_forcing_ratio定义使用Teacher Forcing的比例
        batch_size = trg.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size=self.decoder.output_dim
        # 初始化保存解码器输出的Tensor
        outputs = torch.zeros(
            batch_size,trg_len,trg_vocab_size
        )
        
        # 调用编码器
        hidden, cell = self.encoder(src)
        
        # 解码器的第一个输入应该是起始标识符<sos>
        input=trg[:,0]
        
        for t in range(1,trg_len):           
            output,hidden,cell = self.decoder(input,
                                              hidden,
                                              cell)
            # 保存每次预测结果于outputs
            # outputs = [batch_size,trg_len,trg_vocab_size]
            # output = [batch_size,1,trg_vocab_size]
            outputs[:,t,:] = output
            
            # 随机决定是否使用Teacher Forcing
            teacher_force = random.random() < teacher_forcing_ratio
            
            # output=(batch_size,1,trg_vocab_size)
            top1 = output.argmax(dim=-1) # (batch_size,1)
            
            # if teacher forcing,以真实值作为下一个输入，否则使用预测值
            input = trg[t] if teacher_force else top1
        
        return outputs
        
        
        
        
        